# 🧠 ClipFactory AI Director — v2.3-PRO-STRATEGY
> **VERIFIED BUILD:** Topic-based extraction, Deno-hardened, Montserrat pre-cached.

**Runtime:** Make sure you're on **T4 GPU** (Runtime > Change runtime type > T4 GPU).

### Why can't I see changes?
1. **Browser Cache:** Your browser may have cached the old dashboard. **Press Ctrl+F5** on the Dashboard URL.
2. **Old Projects:** If using the same URL, old results might be loaded. Use the 'Reset' button in the UI.
3. **Ghost Processes:** This launcher now force-kills ghost Python processes on every launch.


In [ ]:
# ─── CELL 0: Google Drive Verification (Hands-free Auto-Mount enabled) ─────────────
import os
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Google Drive is automatically mounted and active!')
else:
    print('⚠️ Google Drive not active. Operating in local ephemeral mode.')


In [ ]:
# ─── CELL 1: Setup & Installation (Run once per session) ───────────────────────

# ── Step 0: Install JS runtimes & Fonts ───────────────────────────────────────
print('🔧 Installing Node.js v20 + Deno (required for yt-dlp)...')
!sudo apt-get update -qq
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - && sudo apt-get install -y nodejs -qq
!sudo ln -sf /usr/bin/nodejs /usr/bin/node
!curl -fsSL https://deno.land/install.sh | sh
import os, urllib.request
os.environ['PATH'] = f"/root/.deno/bin:{os.environ['PATH']}"

print('🎨 Pre-caching Montserrat-Bold font...')
FONT_URL = 'https://github.com/JulietaUla/Montserrat/raw/master/fonts/ttf/Montserrat-Bold.ttf'
os.makedirs('/content/work', exist_ok=True)
if not os.path.exists('/content/work/Montserrat-Bold.ttf'):
    urllib.request.urlretrieve(FONT_URL, '/content/work/Montserrat-Bold.ttf')
    print('✅ Font cached.')

print('✅ Environment check:')
!node --version
!deno --version

# ── Step 1: Clone or Update the repository ─────────────────────────────────────
REPO_URL = 'https://github.com/NiL4gh/clip-factory.git'
REPO_DIR = '/content/clip-factory'

if not os.path.exists(REPO_DIR):
    print(f'🚀 Cloning {REPO_URL}...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('🔄 Repository exists. Pulling latest changes...')
    !cd {REPO_DIR} && git fetch --all && git reset --hard origin/main
    !rm -rf /content/clip_factory/projects/* # FORCE: Clear old project cache
    !rm -rf /content/clip_factory/projects/* # FORCE: Clear old project cache
    !rm -rf /content/clip_factory/projects/* # FORCE: Clear old project cache
    !rm -rf /content/clip_factory/projects/* # FORCE: Clear old project cache

os.chdir(REPO_DIR)

# ── Step 2: Install Python dependencies ────────────────────────────────────────
print('🐍 Installing Python dependencies (~2 min)...')
!yt-dlp -U
!yt-dlp -U
!pip install -r requirements.txt --quiet
!pip uninstall -y yt-dlp && pip install -U https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz
!pip install pyngrok --quiet
print('🧠 Installing LLaMA-CPP (using prebuilt CUDA wheels)...')
!pip install llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124


# ── Step 3: Build Frontend (Next.js Static Export) ─────────────────────────────
print('⚛️  Installing Node dependencies (~1-2 min)...')
frontend_path = os.path.join(REPO_DIR, 'frontend')
if not os.path.exists(os.path.join(frontend_path, 'node_modules')):
    !cd {frontend_path} && npm install --no-audit --no-fund --quiet

print('🏗️  Building production dashboard (optimizing assets)...')
!cd {frontend_path} && NEXT_TELEMETRY_DISABLED=1 npm run build

print('\n✅ Setup complete! All dependencies installed and Dashboard built.')
print('🚀 Now run Cell 2 to launch the dashboard.')

In [ ]:
# ─── CELL 2: Launch ClipFactory.ai ─────────────────────────────────────────────
#@title API Keys (Optional but Recommended)
GEMINI_API_KEY = "" #@param {type:"string"}
import os, sys, time, subprocess, threading, socket, warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

# Try to load from Colab Secrets if field is empty
if not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    except:
        pass

if GEMINI_API_KEY:
    GEMINI_API_KEY = GEMINI_API_KEY.strip()
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    # Write to .env to guarantee FastAPI background threads see it
    with open('/content/clip-factory/.env', 'a') as f:
        f.write(f'\nGEMINI_API_KEY={GEMINI_API_KEY}\n')
    print("✅ Gemini API Key injected and saved to .env!")
!pkill -9 uvicorn || true
!pkill -9 ngrok || true
REPO_DIR = '/content/clip-factory'  # self-contained
time.sleep(2)
if not os.path.exists(REPO_DIR):
    print('❌ ERROR: Repository not found! Please run Cell 1 first.')
else:
    # Kill stale processes on port 8000
    subprocess.run('fuser -k 8000/tcp 2>/dev/null || true', shell=True, stdout=subprocess.DEVNULL)
    time.sleep(1)

    # Check for GPU
    try:
        subprocess.check_output('nvidia-smi')
        print('✅ GPU Detected. LLaMA 3 will run with acceleration.')
    except:
        print('❌ WARNING: No GPU detected! The AI Director will be slow.')

    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Ensure Deno is still in PATH for this session
    os.environ['PATH'] = f"/root/.deno/bin:{os.environ['PATH']}"

    # ── Pre-flight: Verify Deno is actually accessible ──────────────────────────────────────────
    try:
        subprocess.run(['deno', '--version'], capture_output=True, check=True, env=os.environ)
        print('✅ Deno is accessible in Python subprocess.')
    except Exception as e:
        print('❌ CRITICAL: Deno is NOT accessible! yt-dlp will fail. Re-run Cell 1.')
        raise

    # Auto-detect Drive for persistence
    if os.path.exists('/content/drive/MyDrive'):
        BASE = '/content/drive/MyDrive/clip_factory'
        print('✅ Saving models and projects to Google Drive.')
    else:
        BASE = '/content/clip_factory'
        print('⚠️ Using local ephemeral storage.')

    # ── Start FastAPI backend (serves API + static frontend) ──────────────────────
    def run_backend():
        import uvicorn
        uvicorn.run('server.main:app', host='127.0.0.1', port=8000, log_level='info')

    print('🚀 Starting FastAPI backend...')
    backend_thread = threading.Thread(target=run_backend, daemon=True)
    backend_thread.start()

    # Wait for the backend to be ready
    print('⏳ Waiting for port 8000 to be active...')
    ready = False
    for _ in range(30):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('127.0.0.1', 8000)) == 0:
                ready = True
                break
        time.sleep(1)
    
    if not ready:
        print('❌ ERROR: Backend failed to start on port 8000.')
    else:
        print('✅ FastAPI backend is LIVE.')
        
        # ── Create single ngrok tunnel ────────────────────────────────────────────────
        from pyngrok import ngrok
        try:
            ngrok_token = '3EBIL24SVbd1vSEEkgnfWVbESZ6_2PoPMuWkNpq6mK6NpVsfg'
            ngrok.set_auth_token(ngrok_token)
            tunnel = ngrok.connect('127.0.0.1:8000', 'http')
            public_url = tunnel.public_url

            # Inject API URL into the static build
            out_dir = os.path.join(REPO_DIR, 'frontend', 'out')
            if os.path.isdir(out_dir):
                config_js = os.path.join(out_dir, '_config.js')
                with open(config_js, 'w') as f:
                    f.write(f'window.__NEXT_PUBLIC_API_URL = "{public_url}/api";')
                idx_html = os.path.join(out_dir, 'index.html')
                if os.path.exists(idx_html):
                    with open(idx_html, 'r') as f:
                        html = f.read()
                    if '_config.js' not in html:
                        html = html.replace('<head>', '<head><script src="/_config.js"></script>')
                        with open(idx_html, 'w') as f:
                            f.write(html)
                print('✅ API URL auto-injected into dashboard.')

            print(f'\n' + '═' * 60)
            print(f'  🚀  ClipFactory.ai is LIVE')
            print(f'  Dashboard:  {public_url}')
            print(f'  API:        {public_url}/api')
            print('═' * 60)
            print(f'\nOpen the Dashboard URL above — everything is ready to g✅')

            # Keep cell alive
            while True: time.sleep(60)
        except KeyboardInterrupt:
            ngrok.kill()
            print('Tunnel closed.')
